# 강의 04 · 실습 1 — 상태·노드·조건부 엣지 · (5) 고난도 II

## 1. 문제상황

- 사내 IT 헬프데스크에는 한 메일에 질문을 여러 개 적어 보내는 요청자가 많습니다.
- 「와이파이 비밀번호를 알려 달라」는 요청 뒤에 「안내대로 했는데 틀렸다고 나오면 어떻게 하나」, 「회의실도 같은 비밀번호인가」 같은 추가 질문이 같은 메일에 이어집니다.
- 담당자는 요청 본문에 먼저 답하고, 추가 질문마다 안내문을 하나씩 더 씁니다. 추가 질문의 수는 메일마다 달라서 안내문을 몇 개 써야 하는지 정해져 있지 않습니다.
- 긴급 요청은 추가 질문 없이 엔지니어 호출 메시지 하나로 끝나므로, 긴급 요청과 일반 요청의 처리 횟수가 서로 다릅니다.
- 접수 기록에는 요청마다 보낸 안내문 전부를 남겨야 합니다.

## 2. 문제와 목표

- **문제**: 일반 요청은 메일에 적힌 추가 질문의 수만큼 안내문을 더 써야 하고, 그 수는 메일마다 다릅니다. 안내문을 쓰는 일은 한 가지인데 몇 번 반복할지는 실행해 봐야 정해집니다.
- **목표**
  - 요청 한 건과 그 메일에 적힌 추가 질문 목록을 함께 입력합니다.
  - 긴급 요청은 엔지니어 호출 메시지를 만들고, 일반 요청은 첫 질문과 추가 질문 하나하나에 안내문을 차례로 만들어 목록에 쌓습니다. 긴급 요청의 호출 메시지도 `replies`에 한 건 쌓습니다.
    - 긴급 판정의 기준: 서비스가 멈추었거나 여러 사람이 일을 못 하면 긴급, 그 밖은 일반
  - 안내문 노드는 하나만 만들고, 조건부 엣지가 남은 질문이 있는 동안 그 노드로 되돌아가게 합니다.
  - 두 경우 모두 접수 기록까지 진행하는 처리 흐름을 만듭니다.
    - 상태의 키 여섯 개: 요청 본문(`ticket`), 긴급도 판정 결과(`level`), 추가 질문 목록(`followups`), 다음에 답할 질문 번호(`idx`, 0부터 시작하며 0은 요청 본문 자체), 보낸 안내문 목록(`replies`), 기록 여부(`logged`)
    - 접수 요청 두 건과 후속 질문의 문면은 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.
- **목표 달성 여부의 판정 기준**:
  - 추가 질문이 없는 긴급 요청 한 건과 추가 질문이 두 개인 일반 요청 한 건을 입력했을 때,
  - 긴급 요청은 escalate 노드를 한 번 거치고, 일반 요청은 answer 노드를 세 번 거친 뒤 record 노드로 가며,
  - 일반 요청의 최종 상태에서 `replies`의 길이가 3인 것을 실행 결과에서 확인합니다.
    - answer 노드는 질문에 답할 때마다 「[answer] N번째 질문에 답합니다」 줄을, record 노드는 「[기록] …」 줄을 출력하고, 실행이 끝나면 최종 상태(긴급도·안내문 수·기록 여부)를 한 줄로 출력합니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec04_ex01_s5_diagram.svg)

## 4. 단계별 요구사항

이 단에서는 요구사항을 주지 않습니다. 「1. 문제상황」「2. 문제와 목표」「3. 워크플로우 다이어그램」을 보고 요구사항을 번호 목록으로 직접 적습니다. 노드마다 「무엇을 읽고 어느 키에 쓰는가」를 한 항목으로, 엣지 연결과 실행을 각각 한 항목으로 적습니다.

## 5. 코드 골격

이 실습의 코드 골격을 직접 세웁니다. 단계 · 하는 일 · 사용하는 코드 · 대응하는 요구사항 네 칸 표로 적습니다.

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 모델을 준비합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

In [1]:
import os

from dotenv import load_dotenv, find_dotenv
from typing import TypedDict

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")
print("모델 준비를 마쳤습니다.")

모델 준비를 마쳤습니다.


### 단계 ① — 상태 정의 (요구사항 1)

목록 키 두 개(`followups`, `replies`)와 번호 키(`idx`)가 추가됩니다. 노드가 목록 키를 돌려줄 때는 기존 목록에 새 항목을 붙인 새 목록을 돌려줍니다.

In [2]:
class TicketState(TypedDict):
    ticket: str       # 접수된 요청 본문
    level: str        # 긴급도 판정 결과 (긴급 / 일반)
    followups: list   # 후속 질문 목록
    idx: int          # 다음에 답할 질문 번호 (0 = 요청 본문)
    replies: list     # 보낸 안내문 목록
    logged: bool      # 접수 기록 여부


print("상태의 키:", list(TicketState.__annotations__))

상태의 키: ['ticket', 'level', 'followups', 'idx', 'replies', 'logged']


### 단계 ② — 노드 함수 정의 (요구사항 2, 3, 4, 5)

- answer 노드가 여러 번 실행됩니다. 실행할 때마다 `idx`를 보고 어느 질문에 답할지 정하고, 답한 뒤 `idx`를 하나 올립니다.
- `replies`는 덮어쓰지 않고 뒤에 붙입니다.

In [3]:
def triage(state: TicketState) -> dict:
    """요청을 긴급과 일반 중 하나로 판정한다."""
    res = llm.invoke([
        SystemMessage("사내 IT 요청을 긴급, 일반 중 하나로 판정한다. "
                      "서비스가 멈추었거나 여러 사람이 일을 못 하면 긴급이다. "
                      "다른 말 없이 단어 하나만 답한다."),
        HumanMessage(state["ticket"]),
    ])
    return {"level": res.content.strip()}


def escalate(state: TicketState) -> dict:
    """담당 엔지니어를 호출하는 글을 두 문장으로 써서 안내문 목록에 넣는다."""
    res = llm.invoke([
        SystemMessage("긴급 요청을 담당 엔지니어에게 넘기는 호출 글을 두 문장으로 쓴다. "
                      "무엇이 멈추었는지와 무엇을 먼저 확인해야 하는지를 담는다."),
        HumanMessage(state["ticket"]),
    ])
    return {"replies": state["replies"] + [res.content.strip()]}


def answer(state: TicketState) -> dict:
    """idx번째 질문에 답하는 안내문을 두 문장으로 써서 목록 뒤에 붙이고 idx를 올린다."""
    n = state["idx"]
    question = state["ticket"] if n == 0 else state["followups"][n - 1]
    res = llm.invoke([
        SystemMessage("일반 요청에 대해 요청자가 스스로 처리할 수 있는 방법을 "
                      "두 문장으로 안내한다."),
        HumanMessage(question),
    ])
    print(f"    [answer] {n}번째 질문에 답합니다: {question[:24]}...")
    return {"replies": state["replies"] + [res.content.strip()], "idx": n + 1}


def record(state: TicketState) -> dict:
    """처리 결과를 접수 기록에 남긴다 (여기서는 화면 출력으로 대신한다)."""
    print(f"    [기록] 긴급도={state['level']} / 보낸 글 {len(state['replies'])}개")
    return {"logged": True}

### 단계 ③ — 그래프 빌더 생성과 노드 등록 (요구사항 6)

노드는 네 개입니다. 여러 번 실행되는 노드도 등록은 한 번입니다.

In [4]:
g = StateGraph(TicketState)
g.add_node("triage", triage)
g.add_node("escalate", escalate)
g.add_node("answer", answer)
g.add_node("record", record)

print("등록한 노드:", list(g.nodes))

등록한 노드: ['triage', 'escalate', 'answer', 'record']


### 단계 ④ — 엣지 연결 (요구사항 7)

조건부 엣지가 둘입니다. `route_level`은 triage 뒤에서 분기를 고르고, `more_left`는 answer 뒤에서 answer 자신 또는 record를 고릅니다. 판단 함수가 방금 실행한 노드의 이름을 돌려주면 그 노드가 다시 실행됩니다.

In [5]:
def route_level(state: TicketState) -> str:
    """긴급도에 따라 다음에 갈 노드의 이름을 돌려준다."""
    return "escalate" if state["level"] == "긴급" else "answer"


def more_left(state: TicketState) -> str:
    """남은 질문이 있으면 answer로 되돌아가고, 없으면 record로 간다."""
    return "answer" if state["idx"] <= len(state["followups"]) else "record"


g.add_edge(START, "triage")
g.add_conditional_edges("triage", route_level, {"escalate": "escalate", "answer": "answer"})
g.add_conditional_edges("answer", more_left, {"answer": "answer", "record": "record"})
g.add_edge("escalate", "record")
g.add_edge("record", END)

print("고정 엣지 두 개와 조건부 엣지 두 개를 놓았습니다.")

고정 엣지 두 개와 조건부 엣지 두 개를 놓았습니다.


### 단계 ⑤ — 컴파일과 실행 (요구사항 8)

입력에 `followups`·`idx`·`replies`의 시작값을 함께 넣습니다. `stream`의 출력에서 `answer` 줄이 몇 번 출력되는지 셉니다.

In [6]:
graph = g.compile()

TICKETS = [
    ("사내 그룹웨어가 오전 9시부터 접속되지 않습니다. 부서 전체가 결재를 올리지 못하고 있습니다.", []),
    ("노트북 사내 와이파이 비밀번호를 잊어버렸습니다. 다시 알려주실 수 있을까요?",
     ["안내대로 했는데 비밀번호가 틀렸다고 나옵니다.", "회의실 와이파이도 같은 비밀번호인가요?"]),
]

for i, (ticket, followups) in enumerate(TICKETS, 1):
    print(f"=== {i}번 접수: {ticket[:30]}... (후속 질문 {len(followups)}개) ===")
    final = {"ticket": ticket, "followups": followups, "idx": 0, "replies": []}
    for step in graph.stream(dict(final), stream_mode="updates"):
        for node, patch in step.items():
            print(f"  [{node}] -> {list(patch)}")
            final.update(patch)
    print(f"  [최종 상태] level={final['level']!r} replies={len(final['replies'])}개 logged={final.get('logged')}")
    print()

=== 1번 접수: 사내 그룹웨어가 오전 9시부터 접속되지 않습니다. 부서... (후속 질문 0개) ===
  [triage] -> ['level']
  [escalate] -> ['replies']
    [기록] 긴급도=긴급 / 보낸 글 1개
  [record] -> ['logged']
  [최종 상태] level='긴급' replies=1개 logged=True

=== 2번 접수: 노트북 사내 와이파이 비밀번호를 잊어버렸습니다. 다시 ... (후속 질문 2개) ===
  [triage] -> ['level']
    [answer] 0번째 질문에 답합니다: 노트북 사내 와이파이 비밀번호를 잊어버렸습니...
  [answer] -> ['replies', 'idx']
    [answer] 1번째 질문에 답합니다: 안내대로 했는데 비밀번호가 틀렸다고 나옵니다...
  [answer] -> ['replies', 'idx']
    [answer] 2번째 질문에 답합니다: 회의실 와이파이도 같은 비밀번호인가요?...
  [answer] -> ['replies', 'idx']
    [기록] 긴급도=일반 / 보낸 글 3개
  [record] -> ['logged']
  [최종 상태] level='일반' replies=3개 logged=True



## 7. 실행 결과 확인

실행 결과에서 다음 세 가지를 확인합니다.

1. 1번 접수(그룹웨어 장애)는 `triage`, `escalate`, `record`를 한 번씩 거칩니다. 최종 상태의 `replies` 길이는 1입니다.
2. 2번 접수(와이파이 비밀번호, 후속 질문 두 개)는 `triage` 뒤에 `answer` 줄이 세 번 출력되고 그 뒤에 `record`가 출력됩니다. `[answer]` 줄의 질문 번호가 0, 1, 2로 올라갑니다.
3. 2번 접수의 최종 상태에서 `replies` 길이가 3이고 `logged`가 `True`입니다. 같은 노드가 세 번 실행되었지만 등록은 한 번이었습니다.